In [ ]:
# ================================================================================================
# COMPLETE ONE-CELL KAGGLE RESEARCH PIPELINE
# ================================================================================================
#
# PROJECT:
# ViT vs Swin Transformer for Real vs GAN-Generated Face Classification
#
# DATASET:
# xhlulu / 140k Real and Fake Faces
#
# EXPERIMENT:
#   1. Load and visualize dataset
#   2. ViT bake-off: 10,000 train + 2,000 validation
#   3. Swin bake-off: SAME 10,000 + 2,000
#   4. Select winner by validation fake-class ROC-AUC
#   5. Create FRESH winner
#   6. Train winner on 100,000 train images for 2 epochs
#   7. Validate on 20,000 validation images
#   8. Use untouched 20,000-image test set ONCE
#   9. Generate research tables, graphs, predictions and model
#
# IMPORTANT:
#   - Pure PyTorch + timm
#   - No Hugging Face Trainer
#   - No DataParallel
#   - GPU 0 only
#   - Fake = positive forensic class
#   - All results saved to /kaggle/working/vit_vs_swin_research/
#
# ================================================================================================


# ================================================================================================
# SECTION 0 — INSTALL / IMPORT
# ================================================================================================

import sys
import subprocess

print("=" * 105)
print("SECTION 0 — INSTALLING / VERIFYING DEPENDENCIES")
print("=" * 105)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "timm>=1.0.20,<1.1",
    "scikit-learn"
])

import os
import gc
import json
import math
import time
import random
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import timm

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    cohen_kappa_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    classification_report
)

from sklearn.calibration import calibration_curve

print("✅ Dependencies ready.")


# ================================================================================================
# SECTION 1 — CONFIGURATION
# ================================================================================================

SEED = 42

IMAGE_SIZE = 224

# Architecture bake-off
BAKE_TRAIN_SIZE = 10_000
BAKE_VAL_SIZE = 2_000
BAKE_EPOCHS = 1

# Full selected-model training
FULL_EPOCHS = 2

# Batches
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUMULATION = 2

# Optimizer
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0

NUM_WORKERS = 2

# Bootstrap confidence intervals
BOOTSTRAP_ITERATIONS = 500


OUTPUT_ROOT = Path(
    "/kaggle/working/vit_vs_swin_research"
)

DATA_DIR = OUTPUT_ROOT / "00_Dataset"
VIT_DIR = OUTPUT_ROOT / "01_ViT"
SWIN_DIR = OUTPUT_ROOT / "02_Swin"
COMPARE_DIR = OUTPUT_ROOT / "03_Architecture_Comparison"
FINAL_DIR = OUTPUT_ROOT / "04_Final_Winner"


# Clean partial output from previous failed attempts.
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

for directory in [
    OUTPUT_ROOT,
    DATA_DIR,
    VIT_DIR,
    SWIN_DIR,
    COMPARE_DIR,
    FINAL_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("\n✅ Configuration loaded.")


# ================================================================================================
# SECTION 2 — REPRODUCIBILITY
# ================================================================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

print("✅ Random seed:", SEED)


# ================================================================================================
# SECTION 3 — HARDWARE
# ================================================================================================

print("\n" + "=" * 105)
print("SECTION 3 — HARDWARE")
print("=" * 105)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())

for gpu_id in range(torch.cuda.device_count()):
    print(
        f"GPU {gpu_id}:",
        torch.cuda.get_device_name(gpu_id)
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. "
        "Kaggle → Settings → Accelerator → GPU."
    )

# Deliberately GPU 0 only.
DEVICE = torch.device("cuda:0")

print("\nTraining device:", DEVICE)
print("✅ Single-GPU training enabled.")


# ================================================================================================
# SECTION 4 — FIND THE REAL DATASET
# ================================================================================================

print("\n" + "=" * 105)
print("SECTION 4 — LOCATING THE REAL 140K DATASET")
print("=" * 105)


def is_correct_dataset_root(path):
    """
    A valid dataset root must contain:
      train/fake
      train/real
      valid(or validation or val)/fake
      valid(or validation or val)/real
      test/fake
      test/real
    """

    path = Path(path)

    train_ok = (
        (path / "train" / "fake").is_dir()
        and
        (path / "train" / "real").is_dir()
    )

    test_ok = (
        (path / "test" / "fake").is_dir()
        and
        (path / "test" / "real").is_dir()
    )

    validation_ok = False

    for validation_name in [
        "valid",
        "validation",
        "val"
    ]:
        if (
            (path / validation_name / "fake").is_dir()
            and
            (path / validation_name / "real").is_dir()
        ):
            validation_ok = True
            break

    return train_ok and validation_ok and test_ok


# Normal Kaggle locations.
candidate_paths = [
    Path(
        "/kaggle/input/140k-real-and-fake-faces/"
        "real_vs_fake/real-vs-fake"
    ),

    Path(
        "/kaggle/input/datasets/xhlulu/"
        "140k-real-and-fake-faces/"
        "real_vs_fake/real-vs-fake"
    ),

    Path(
        "/kaggle/input/140k-real-and-fake-faces/"
        "real_vs_fake"
    ),

    Path(
        "/kaggle/input/datasets/xhlulu/"
        "140k-real-and-fake-faces/"
        "real_vs_fake"
    )
]


DATASET_ROOT = None


for candidate in candidate_paths:

    if is_correct_dataset_root(candidate):

        DATASET_ROOT = candidate
        break


# Automatic fallback search.
if DATASET_ROOT is None:

    print(
        "Standard path not found. "
        "Searching /kaggle/input automatically..."
    )

    for root, dirs, files in os.walk(
        "/kaggle/input"
    ):

        root_path = Path(root)

        if is_correct_dataset_root(
            root_path
        ):

            DATASET_ROOT = root_path
            break


if DATASET_ROOT is None:

    print("\nInputs visible to Kaggle:")

    for item in Path(
        "/kaggle/input"
    ).iterdir():

        print(" •", item)

    raise FileNotFoundError(
        "\nThe REAL dataset is still not visible.\n"
        "The attached input must contain real_vs_fake/train, valid and test image folders."
    )


TRAIN_PATH = DATASET_ROOT / "train"
TEST_PATH = DATASET_ROOT / "test"

if (DATASET_ROOT / "valid").is_dir():

    VAL_PATH = DATASET_ROOT / "valid"

elif (DATASET_ROOT / "validation").is_dir():

    VAL_PATH = DATASET_ROOT / "validation"

else:

    VAL_PATH = DATASET_ROOT / "val"


print("\n✅ DATASET FOUND SUCCESSFULLY")
print("Dataset root:")
print(DATASET_ROOT)

print("\nTrain:")
print(TRAIN_PATH)

print("\nValidation:")
print(VAL_PATH)

print("\nTest:")
print(TEST_PATH)


# ================================================================================================
# SECTION 5 — IMAGE PREPROCESSING
# ================================================================================================

print("\n" + "=" * 105)
print("SECTION 5 — IMAGE PREPROCESSING")
print("=" * 105)


IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225
)


# Identical augmentation strategy for both architectures.
train_transform = transforms.Compose([

    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.90, 1.00),
        ratio=(0.95, 1.05)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])


# Deterministic validation/test pipeline.
eval_transform = transforms.Compose([

    transforms.Resize(
        256
    ),

    transforms.CenterCrop(
        IMAGE_SIZE
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])


print(
    f"Image resolution: "
    f"{IMAGE_SIZE} × {IMAGE_SIZE}"
)

print(
    "✅ Preprocessing configured."
)


# ================================================================================================
# SECTION 6 — LOAD DATASETS
# ================================================================================================

print("\n" + "=" * 105)
print("SECTION 6 — LOADING DATASETS")
print("=" * 105)


raw_train = datasets.ImageFolder(
    TRAIN_PATH
)

raw_val = datasets.ImageFolder(
    VAL_PATH
)

raw_test = datasets.ImageFolder(
    TEST_PATH
)


train_dataset = datasets.ImageFolder(
    TRAIN_PATH,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_PATH,
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    TEST_PATH,
    transform=eval_transform
)


CLASSES = raw_train.classes
CLASS_TO_ID = raw_train.class_to_idx


if "fake" not in CLASS_TO_ID:
    raise RuntimeError(
        "The 'fake' class is missing."
    )

if "real" not in CLASS_TO_ID:
    raise RuntimeError(
        "The 'real' class is missing."
    )


FAKE_ID = CLASS_TO_ID["fake"]
REAL_ID = CLASS_TO_ID["real"]


print("Classes:", CLASSES)
print("Class mapping:", CLASS_TO_ID)

print()

print(
    f"Training images   : "
    f"{len(raw_train):,}"
)

print(
    f"Validation images : "
    f"{len(raw_val):,}"
)

print(
    f"Test images       : "
    f"{len(raw_test):,}"
)

print()

print("FAKE_ID:", FAKE_ID)
print("REAL_ID:", REAL_ID)
print("Positive forensic class: FAKE")


# Sanity checks.
if len(raw_train) < 90_000:
    raise RuntimeError(
        f"Unexpected training-set size: {len(raw_train)}"
    )

if len(raw_val) < 15_000:
    raise RuntimeError(
        f"Unexpected validation-set size: {len(raw_val)}"
    )

if len(raw_test) < 15_000:
    raise RuntimeError(
        f"Unexpected test-set size: {len(raw_test)}"
    )


print("\n✅ Dataset-size sanity checks passed.")


# ================================================================================================
# SECTION 7 — DATASET CLASS COUNTS
# ================================================================================================

def get_class_counts(dataset):

    targets = np.asarray(
        dataset.targets,
        dtype=np.int64
    )

    counts = {}

    for class_name, class_id in (
        dataset.class_to_idx.items()
    ):

        counts[class_name] = int(
            np.sum(
                targets == class_id
            )
        )

    return counts


train_counts = get_class_counts(raw_train)
val_counts = get_class_counts(raw_val)
test_counts = get_class_counts(raw_test)


print("\nTraining counts:")
print(train_counts)

print("\nValidation counts:")
print(val_counts)

print("\nTest counts:")
print(test_counts)


dataset_rows = []

for split_name, counts in [
    ("Train", train_counts),
    ("Validation", val_counts),
    ("Test", test_counts)
]:

    for class_name in CLASSES:

        dataset_rows.append({
            "split": split_name,
            "class": class_name,
            "count": counts[class_name]
        })


dataset_summary_df = pd.DataFrame(
    dataset_rows
)

dataset_summary_df.to_csv(
    DATA_DIR / "dataset_distribution.csv",
    index=False
)


# ================================================================================================
# SECTION 8 — VISUALIZE DATASET DISTRIBUTION
# ================================================================================================

fig, ax = plt.subplots(
    figsize=(9, 5)
)

splits = [
    "Train",
    "Validation",
    "Test"
]

x = np.arange(
    len(splits)
)

width = 0.35


for class_number, class_name in enumerate(
    CLASSES
):

    values = []

    for split_name in splits:

        value = dataset_summary_df[
            (
                dataset_summary_df["split"]
                ==
                split_name
            )
            &
            (
                dataset_summary_df["class"]
                ==
                class_name
            )
        ]["count"].iloc[0]

        values.append(
            int(value)
        )


    positions = (
        x
        +
        (
            class_number
            -
            0.5
        )
        *
        width
    )


    bars = ax.bar(
        positions,
        values,
        width,
        label=class_name.title()
    )


    for bar, value in zip(
        bars,
        values
    ):

        ax.text(
            bar.get_x()
            +
            bar.get_width() / 2,
            value,
            f"{value:,}",
            ha="center",
            va="bottom",
            fontsize=9
        )


ax.set_xticks(x)
ax.set_xticklabels(splits)

ax.set_ylabel(
    "Number of Images"
)

ax.set_title(
    "140k Real and Fake Faces — Dataset Distribution"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    DATA_DIR / "dataset_distribution.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 9 — VISUALIZE SAMPLE REAL/FAKE IMAGES
# ================================================================================================

train_targets_array = np.asarray(
    raw_train.targets,
    dtype=np.int64
)


fig, axes = plt.subplots(
    2,
    5,
    figsize=(15, 6)
)


for row, class_name in enumerate([
    "fake",
    "real"
]):

    class_id = CLASS_TO_ID[
        class_name
    ]

    sample_indices = np.flatnonzero(
        train_targets_array
        ==
        class_id
    )[:5]


    for column, sample_index in enumerate(
        sample_indices
    ):

        image_path, _ = raw_train.samples[
            int(sample_index)
        ]

        with Image.open(
            image_path
        ) as image:

            image = image.convert("RGB")

            axes[
                row,
                column
            ].imshow(
                image
            )

        axes[
            row,
            column
        ].set_title(
            class_name.title()
        )

        axes[
            row,
            column
        ].axis("off")


fig.suptitle(
    "Example Real and GAN-Generated Face Images",
    fontsize=16
)

fig.tight_layout()

fig.savefig(
    DATA_DIR / "dataset_example_images.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 10 — CREATE IDENTICAL STRATIFIED BAKE-OFF SUBSETS
# ================================================================================================

def make_stratified_indices(
    targets,
    requested_size
):

    targets = np.asarray(
        targets,
        dtype=np.int64
    )

    indices = np.arange(
        len(targets)
    )

    if requested_size >= len(indices):
        return indices.tolist()


    selected, _ = train_test_split(
        indices,
        train_size=requested_size,
        random_state=SEED,
        stratify=targets
    )

    return selected.tolist()


bake_train_indices = make_stratified_indices(
    raw_train.targets,
    BAKE_TRAIN_SIZE
)

bake_val_indices = make_stratified_indices(
    raw_val.targets,
    BAKE_VAL_SIZE
)


bake_train_dataset = Subset(
    train_dataset,
    bake_train_indices
)

bake_val_dataset = Subset(
    val_dataset,
    bake_val_indices
)


print("\n" + "=" * 105)
print("SECTION 10 — ARCHITECTURE BAKE-OFF")
print("=" * 105)

print(
    f"Training subset   : "
    f"{len(bake_train_dataset):,}"
)

print(
    f"Validation subset : "
    f"{len(bake_val_dataset):,}"
)

print(
    f"Epochs/model      : "
    f"{BAKE_EPOCHS}"
)


# ================================================================================================
# SECTION 11 — DATALOADERS
# ================================================================================================

def seed_worker(worker_id):

    worker_seed = (
        SEED + worker_id
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


def make_loader(
    dataset,
    batch_size,
    shuffle
):

    generator = torch.Generator()

    generator.manual_seed(
        SEED
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(
            NUM_WORKERS > 0
        ),
        worker_init_fn=seed_worker,
        generator=generator
    )


# ================================================================================================
# SECTION 12 — SELECT TIMM ViT + SWIN
# ================================================================================================

print("\n" + "=" * 105)
print("SECTION 12 — SELECTING PRETRAINED TRANSFORMERS")
print("=" * 105)


available_models = set(
    timm.list_models(
        pretrained=True
    )
)


def select_model(
    candidates,
    fallback_pattern,
    architecture_name
):

    for candidate in candidates:

        if candidate in available_models:

            print(
                f"✅ {architecture_name}: "
                f"{candidate}"
            )

            return candidate


    fallback_models = timm.list_models(
        fallback_pattern,
        pretrained=True
    )


    if len(fallback_models) > 0:

        selected = fallback_models[0]

        print(
            f"✅ {architecture_name} fallback: "
            f"{selected}"
        )

        return selected


    raise RuntimeError(
        f"No compatible pretrained "
        f"{architecture_name} model found."
    )


VIT_MODEL_NAME = select_model(
    candidates=[
        "vit_base_patch16_224.augreg2_in21k_ft_in1k",
        "vit_base_patch16_224.augreg_in21k_ft_in1k",
        "vit_base_patch16_224.orig_in21k_ft_in1k"
    ],
    fallback_pattern="*vit_base_patch16_224*",
    architecture_name="ViT"
)


SWIN_MODEL_NAME = select_model(
    candidates=[
        "swin_tiny_patch4_window7_224.ms_in1k",
        "swin_tiny_patch4_window7_224"
    ],
    fallback_pattern="*swin_tiny_patch4_window7_224*",
    architecture_name="Swin"
)


MODEL_NAMES = {
    "ViT": VIT_MODEL_NAME,
    "Swin": SWIN_MODEL_NAME
}


# ================================================================================================
# SECTION 13 — MODEL CREATION
# ================================================================================================

def create_model(model_name):

    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=2
    )

    return model


# ================================================================================================
# SECTION 14 — PARAMETER COMPARISON
# ================================================================================================

complexity_rows = []


for architecture in [
    "ViT",
    "Swin"
]:

    temp_model = create_model(
        MODEL_NAMES[
            architecture
        ]
    )


    total_parameters = sum(
        parameter.numel()
        for parameter
        in temp_model.parameters()
    )


    trainable_parameters = sum(
        parameter.numel()
        for parameter
        in temp_model.parameters()
        if parameter.requires_grad
    )


    complexity_rows.append({
        "model": architecture,
        "checkpoint": MODEL_NAMES[architecture],
        "parameters": total_parameters,
        "parameters_millions":
            total_parameters / 1_000_000,
        "trainable_parameters":
            trainable_parameters
    })


    print(
        f"{architecture}: "
        f"{total_parameters / 1e6:.2f}M parameters"
    )


    del temp_model
    gc.collect()


complexity_df = pd.DataFrame(
    complexity_rows
)

complexity_df.to_csv(
    COMPARE_DIR / "model_complexity.csv",
    index=False
)


fig, ax = plt.subplots(
    figsize=(7, 5)
)

bars = ax.bar(
    complexity_df["model"],
    complexity_df[
        "parameters_millions"
    ]
)

ax.set_ylabel(
    "Parameters (Millions)"
)

ax.set_title(
    "ViT vs Swin Model Size"
)

for bar, value in zip(
    bars,
    complexity_df[
        "parameters_millions"
    ]
):

    ax.text(
        bar.get_x()
        +
        bar.get_width() / 2,
        value,
        f"{value:.1f}M",
        ha="center",
        va="bottom"
    )

fig.tight_layout()

fig.savefig(
    COMPARE_DIR / "parameter_comparison.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 15 — METRICS
# ================================================================================================

def numpy_softmax(logits):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    logits = (
        logits
        -
        np.max(
            logits,
            axis=1,
            keepdims=True
        )
    )

    exp_logits = np.exp(
        logits
    )

    return (
        exp_logits
        /
        exp_logits.sum(
            axis=1,
            keepdims=True
        )
    )


def calculate_metrics(
    logits,
    labels
):

    labels = np.asarray(
        labels,
        dtype=np.int64
    )


    probabilities = numpy_softmax(
        logits
    )


    predictions = np.argmax(
        probabilities,
        axis=1
    )


    # IMPORTANT:
    # Dataset class IDs:
    # fake = 0
    # real = 1
    #
    # For forensic binary metrics:
    # fake becomes positive binary class 1.

    fake_probability = probabilities[
        :,
        FAKE_ID
    ]


    binary_true = (
        labels
        ==
        FAKE_ID
    ).astype(
        np.int64
    )


    binary_prediction = (
        predictions
        ==
        FAKE_ID
    ).astype(
        np.int64
    )


    cm = confusion_matrix(
        binary_true,
        binary_prediction,
        labels=[
            0,
            1
        ]
    )


    tn, fp, fn, tp = cm.ravel()


    specificity_real = (
        tn
        /
        (tn + fp)
        if (
            tn + fp
        ) > 0
        else 0.0
    )


    brier = float(
        np.mean(
            (
                fake_probability
                -
                binary_true
            )
            ** 2
        )
    )


    metrics = {
        "accuracy":
            float(
                accuracy_score(
                    labels,
                    predictions
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    labels,
                    predictions
                )
            ),

        "precision_fake":
            float(
                precision_score(
                    binary_true,
                    binary_prediction,
                    zero_division=0
                )
            ),

        "recall_fake":
            float(
                recall_score(
                    binary_true,
                    binary_prediction,
                    zero_division=0
                )
            ),

        "specificity_real":
            float(
                specificity_real
            ),

        "f1_fake":
            float(
                f1_score(
                    binary_true,
                    binary_prediction,
                    zero_division=0
                )
            ),

        "auc_fake":
            float(
                roc_auc_score(
                    binary_true,
                    fake_probability
                )
            ),

        "average_precision_fake":
            float(
                average_precision_score(
                    binary_true,
                    fake_probability
                )
            ),

        "kappa":
            float(
                cohen_kappa_score(
                    labels,
                    predictions
                )
            ),

        "mcc":
            float(
                matthews_corrcoef(
                    binary_true,
                    binary_prediction
                )
            ),

        "brier_fake":
            brier
    }


    return {
        "metrics":
            metrics,

        "labels":
            labels,

        "predictions":
            predictions,

        "probabilities":
            probabilities,

        "fake_probability":
            fake_probability,

        "binary_true":
            binary_true,

        "binary_prediction":
            binary_prediction,

        "confusion_matrix":
            cm
    }


# ================================================================================================
# SECTION 16 — EVALUATION
# ================================================================================================

@torch.inference_mode()
def evaluate_model(
    model,
    loader,
    criterion,
    description
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    logits_collection = []
    labels_collection = []


    for images, labels in tqdm(
        loader,
        desc=description,
        leave=False
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True
        ):

            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels
            )


        batch_size = images.size(0)

        total_loss += (
            loss.item()
            *
            batch_size
        )

        total_samples += batch_size


        logits_collection.append(
            logits
            .detach()
            .float()
            .cpu()
            .numpy()
        )

        labels_collection.append(
            labels
            .detach()
            .cpu()
            .numpy()
        )


    all_logits = np.concatenate(
        logits_collection
    )

    all_labels = np.concatenate(
        labels_collection
    )


    result = calculate_metrics(
        all_logits,
        all_labels
    )


    result["loss"] = (
        total_loss
        /
        total_samples
    )


    result["logits"] = (
        all_logits
    )


    return result


# ================================================================================================
# SECTION 17 — CHECKPOINT LOAD
# ================================================================================================

def safe_load_state_dict(path):

    try:

        return torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

    except TypeError:

        return torch.load(
            path,
            map_location="cpu"
        )


# ================================================================================================
# SECTION 18 — TRAINING
# ================================================================================================

def train_architecture(
    architecture,
    model_name,
    training_data,
    validation_data,
    epochs,
    output_directory,
    keep_model=False
):

    print("\n" + "=" * 105)
    print(f"TRAINING {architecture}")
    print("=" * 105)

    print("Checkpoint:", model_name)
    print(
        "Training images:",
        f"{len(training_data):,}"
    )
    print(
        "Validation images:",
        f"{len(validation_data):,}"
    )
    print("Epochs:", epochs)


    output_directory.mkdir(
        parents=True,
        exist_ok=True
    )


    gc.collect()
    torch.cuda.empty_cache()


    model = create_model(
        model_name
    ).to(
        DEVICE
    )


    criterion = nn.CrossEntropyLoss()


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )


    train_loader = make_loader(
        training_data,
        TRAIN_BATCH_SIZE,
        shuffle=True
    )


    validation_loader = make_loader(
        validation_data,
        EVAL_BATCH_SIZE,
        shuffle=False
    )


    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True
    )


    history = {
        "epoch": [],
        "train_loss": [],
        "validation_loss": [],
        "validation_accuracy": [],
        "validation_balanced_accuracy": [],
        "validation_f1_fake": [],
        "validation_auc_fake": [],
        "step": [],
        "step_loss": []
    }


    best_auc = -np.inf

    best_checkpoint = (
        output_directory
        /
        "best_model_state_dict.pt"
    )


    global_step = 0

    start_time = time.time()


    for epoch in range(
        1,
        epochs + 1
    ):

        print(
            "\n"
            +
            "-" * 105
        )

        print(
            f"{architecture} "
            f"EPOCH {epoch}/{epochs}"
        )

        print(
            "-" * 105
        )


        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )


        running_loss = 0.0
        running_samples = 0


        progress = tqdm(
            train_loader,
            desc=(
                f"{architecture} "
                f"Epoch {epoch}/{epochs}"
            )
        )


        for batch_index, (
            images,
            labels
        ) in enumerate(
            progress
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )


            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True
            ):

                logits = model(
                    images
                )

                raw_loss = criterion(
                    logits,
                    labels
                )

                loss = (
                    raw_loss
                    /
                    GRAD_ACCUMULATION
                )


            scaler.scale(
                loss
            ).backward()


            should_update = (
                (
                    (
                        batch_index + 1
                    )
                    %
                    GRAD_ACCUMULATION
                )
                ==
                0
                or
                (
                    batch_index + 1
                )
                ==
                len(train_loader)
            )


            if should_update:

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP_NORM
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

                global_step += 1

                history["step"].append(
                    global_step
                )

                history["step_loss"].append(
                    float(
                        raw_loss.item()
                    )
                )


            batch_size = images.size(
                0
            )

            running_loss += (
                raw_loss.item()
                *
                batch_size
            )

            running_samples += (
                batch_size
            )


            progress.set_postfix(
                loss=(
                    f"{raw_loss.item():.4f}"
                )
            )


        epoch_training_loss = (
            running_loss
            /
            running_samples
        )


        validation_result = evaluate_model(
            model,
            validation_loader,
            criterion,
            description=(
                f"{architecture} Validation"
            )
        )


        metrics = validation_result[
            "metrics"
        ]


        history["epoch"].append(
            epoch
        )

        history["train_loss"].append(
            epoch_training_loss
        )

        history[
            "validation_loss"
        ].append(
            validation_result["loss"]
        )

        history[
            "validation_accuracy"
        ].append(
            metrics["accuracy"]
        )

        history[
            "validation_balanced_accuracy"
        ].append(
            metrics[
                "balanced_accuracy"
            ]
        )

        history[
            "validation_f1_fake"
        ].append(
            metrics["f1_fake"]
        )

        history[
            "validation_auc_fake"
        ].append(
            metrics["auc_fake"]
        )


        print(
            f"\nTrain loss            : "
            f"{epoch_training_loss:.6f}"
        )

        print(
            f"Validation loss       : "
            f"{validation_result['loss']:.6f}"
        )

        print(
            f"Accuracy              : "
            f"{metrics['accuracy']:.6f}"
        )

        print(
            f"Balanced accuracy     : "
            f"{metrics['balanced_accuracy']:.6f}"
        )

        print(
            f"Fake precision        : "
            f"{metrics['precision_fake']:.6f}"
        )

        print(
            f"Fake recall           : "
            f"{metrics['recall_fake']:.6f}"
        )

        print(
            f"Real specificity      : "
            f"{metrics['specificity_real']:.6f}"
        )

        print(
            f"Fake F1               : "
            f"{metrics['f1_fake']:.6f}"
        )

        print(
            f"Fake ROC-AUC          : "
            f"{metrics['auc_fake']:.6f}"
        )

        print(
            f"Fake PR-AUC           : "
            f"{metrics['average_precision_fake']:.6f}"
        )

        print(
            f"Cohen Kappa           : "
            f"{metrics['kappa']:.6f}"
        )

        print(
            f"MCC                   : "
            f"{metrics['mcc']:.6f}"
        )


        if (
            metrics["auc_fake"]
            >
            best_auc
        ):

            best_auc = (
                metrics["auc_fake"]
            )

            torch.save(
                model.state_dict(),
                best_checkpoint
            )

            print(
                "✅ Best validation checkpoint saved."
            )


    training_seconds = (
        time.time()
        -
        start_time
    )


    # Reload our exact PyTorch state_dict.
    best_state = safe_load_state_dict(
        best_checkpoint
    )

    model.load_state_dict(
        best_state,
        strict=True
    )

    model = model.to(
        DEVICE
    )


    final_validation = evaluate_model(
        model,
        validation_loader,
        criterion,
        description=(
            f"{architecture} Best Validation"
        )
    )


    metrics = final_validation[
        "metrics"
    ]


    print("\n" + "=" * 105)
    print(
        f"{architecture} BEST VALIDATION RESULTS"
    )
    print("=" * 105)


    for name, value in metrics.items():

        print(
            f"{name:28s}: "
            f"{value:.6f}"
        )


    print(
        f"Training time (minutes)     : "
        f"{training_seconds / 60:.2f}"
    )


    epoch_history = pd.DataFrame({
        "epoch":
            history["epoch"],

        "train_loss":
            history["train_loss"],

        "validation_loss":
            history[
                "validation_loss"
            ],

        "validation_accuracy":
            history[
                "validation_accuracy"
            ],

        "validation_balanced_accuracy":
            history[
                "validation_balanced_accuracy"
            ],

        "validation_f1_fake":
            history[
                "validation_f1_fake"
            ],

        "validation_auc_fake":
            history[
                "validation_auc_fake"
            ]
    })


    epoch_history.to_csv(
        output_directory
        /
        "epoch_history.csv",
        index=False
    )


    step_history = pd.DataFrame({
        "step":
            history["step"],

        "loss":
            history["step_loss"]
    })


    step_history.to_csv(
        output_directory
        /
        "step_loss.csv",
        index=False
    )


    run_information = {
        "architecture":
            architecture,

        "checkpoint":
            model_name,

        "epochs":
            epochs,

        "training_images":
            len(training_data),

        "validation_images":
            len(validation_data),

        "training_seconds":
            training_seconds,

        "metrics":
            metrics
    }


    with open(
        output_directory
        /
        "results.json",
        "w"
    ) as file:

        json.dump(
            run_information,
            file,
            indent=2
        )


    result = {
        "architecture":
            architecture,

        "model_name":
            model_name,

        "checkpoint":
            str(
                best_checkpoint
            ),

        "history":
            history,

        "validation":
            final_validation,

        "metrics":
            metrics,

        "training_seconds":
            training_seconds
    }


    if keep_model:

        result["model"] = model

    else:

        model = model.to(
            "cpu"
        )

        del model

        gc.collect()

        torch.cuda.empty_cache()


    return result


# ================================================================================================
# SECTION 19 — VISUALIZATION FUNCTIONS
# ================================================================================================

def plot_training_loss(
    result,
    output_path
):

    steps = np.asarray(
        result[
            "history"
        ]["step"]
    )

    losses = np.asarray(
        result[
            "history"
        ]["step_loss"]
    )


    fig, ax = plt.subplots(
        figsize=(9, 5)
    )


    if len(losses) >= 20:

        window = min(
            30,
            len(losses)
        )

        smoothed = np.convolve(
            losses,
            np.ones(window) / window,
            mode="valid"
        )

        plot_steps = steps[
            window - 1:
        ]

        ax.plot(
            plot_steps,
            smoothed
        )

    else:

        ax.plot(
            steps,
            losses
        )


    ax.set_xlabel(
        "Optimizer Update"
    )

    ax.set_ylabel(
        "Cross-Entropy Loss"
    )

    ax.set_title(
        f"{result['architecture']} "
        "Training Loss"
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=180
    )

    plt.show()
    plt.close(fig)


def plot_confusion_matrix(
    cm,
    title,
    output_path,
    normalized=False
):

    display = np.asarray(
        cm,
        dtype=np.float64
    )


    if normalized:

        denominator = display.sum(
            axis=1,
            keepdims=True
        )

        denominator[
            denominator == 0
        ] = 1

        display = (
            display
            /
            denominator
        )


    fig, ax = plt.subplots(
        figsize=(6, 5)
    )


    image = ax.imshow(
        display
    )


    # Binary forensic representation:
    # 0 = Real
    # 1 = Fake

    ax.set_xticks([
        0,
        1
    ])

    ax.set_yticks([
        0,
        1
    ])

    ax.set_xticklabels([
        "Real",
        "Fake"
    ])

    ax.set_yticklabels([
        "Real",
        "Fake"
    ])

    ax.set_xlabel(
        "Predicted Class"
    )

    ax.set_ylabel(
        "Actual Class"
    )

    ax.set_title(
        title
    )


    for row in range(2):

        for column in range(2):

            if normalized:

                text = (
                    f"{display[row, column]:.2%}"
                )

            else:

                text = (
                    f"{int(cm[row, column]):,}"
                )


            ax.text(
                column,
                row,
                text,
                ha="center",
                va="center",
                fontsize=12
            )


    fig.colorbar(
        image,
        ax=ax
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=180
    )

    plt.show()
    plt.close(fig)


def plot_probability_distribution(
    evaluation,
    title,
    output_path
):

    probabilities = evaluation[
        "fake_probability"
    ]

    binary_true = evaluation[
        "binary_true"
    ]


    fig, ax = plt.subplots(
        figsize=(9, 5)
    )


    ax.hist(
        probabilities[
            binary_true == 1
        ],
        bins=40,
        alpha=0.6,
        label="Actual Fake"
    )


    ax.hist(
        probabilities[
            binary_true == 0
        ],
        bins=40,
        alpha=0.6,
        label="Actual Real"
    )


    ax.set_xlabel(
        "Predicted Probability of Fake"
    )

    ax.set_ylabel(
        "Number of Images"
    )

    ax.set_title(
        title
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=180
    )

    plt.show()
    plt.close(fig)


def plot_misclassified_examples(
    evaluation,
    raw_dataset,
    original_indices,
    title,
    output_path,
    maximum=12
):

    predictions = evaluation[
        "predictions"
    ]

    true_labels = evaluation[
        "labels"
    ]

    fake_probability = evaluation[
        "fake_probability"
    ]


    incorrect = np.where(
        predictions
        !=
        true_labels
    )[0]


    if len(incorrect) == 0:

        print(
            "No misclassified examples for:",
            title
        )

        return


    confidence = np.where(
        predictions
        ==
        FAKE_ID,
        fake_probability,
        1
        -
        fake_probability
    )


    incorrect = sorted(
        incorrect,
        key=lambda i:
            confidence[i],
        reverse=True
    )[:maximum]


    columns = 4

    rows = math.ceil(
        len(incorrect)
        /
        columns
    )


    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(
            14,
            rows * 3.5
        ),
        squeeze=False
    )


    axes = axes.flatten()


    for axis in axes:
        axis.axis("off")


    for plot_index, position in enumerate(
        incorrect
    ):

        original_index = original_indices[
            position
        ]

        image_path, _ = raw_dataset.samples[
            original_index
        ]


        with Image.open(
            image_path
        ) as image:

            image = image.convert(
                "RGB"
            )

            axes[
                plot_index
            ].imshow(
                image
            )


        true_name = CLASSES[
            int(
                true_labels[
                    position
                ]
            )
        ]


        predicted_name = CLASSES[
            int(
                predictions[
                    position
                ]
            )
        ]


        axes[
            plot_index
        ].set_title(
            f"True: {true_name}\n"
            f"Pred: {predicted_name}\n"
            f"P(fake): "
            f"{fake_probability[position]:.3f}"
        )


        axes[
            plot_index
        ].axis("off")


    fig.suptitle(
        title,
        fontsize=15
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=180
    )

    plt.show()
    plt.close(fig)


# ================================================================================================
# SECTION 20 — ViT BAKE-OFF
# ================================================================================================

print("\n" + "#" * 105)
print("SECTION 20 — VISION TRANSFORMER (ViT) BAKE-OFF")
print("#" * 105)


vit_result = train_architecture(
    architecture="ViT",
    model_name=VIT_MODEL_NAME,
    training_data=bake_train_dataset,
    validation_data=bake_val_dataset,
    epochs=BAKE_EPOCHS,
    output_directory=VIT_DIR,
    keep_model=False
)


plot_training_loss(
    vit_result,
    VIT_DIR / "training_loss.png"
)


plot_confusion_matrix(
    vit_result[
        "validation"
    ]["confusion_matrix"],
    "ViT Validation Confusion Matrix",
    VIT_DIR / "confusion_matrix.png"
)


plot_confusion_matrix(
    vit_result[
        "validation"
    ]["confusion_matrix"],
    "ViT Normalized Validation Confusion Matrix",
    VIT_DIR / "confusion_matrix_normalized.png",
    normalized=True
)


plot_probability_distribution(
    vit_result[
        "validation"
    ],
    "ViT Validation Prediction Distribution",
    VIT_DIR / "prediction_distribution.png"
)


plot_misclassified_examples(
    vit_result[
        "validation"
    ],
    raw_val,
    bake_val_indices,
    "ViT Highest-Confidence Validation Errors",
    VIT_DIR / "misclassified_examples.png"
)


gc.collect()
torch.cuda.empty_cache()


# ================================================================================================
# SECTION 21 — SWIN BAKE-OFF
# ================================================================================================

print("\n" + "#" * 105)
print("SECTION 21 — SWIN TRANSFORMER BAKE-OFF")
print("#" * 105)


swin_result = train_architecture(
    architecture="Swin",
    model_name=SWIN_MODEL_NAME,
    training_data=bake_train_dataset,
    validation_data=bake_val_dataset,
    epochs=BAKE_EPOCHS,
    output_directory=SWIN_DIR,
    keep_model=False
)


plot_training_loss(
    swin_result,
    SWIN_DIR / "training_loss.png"
)


plot_confusion_matrix(
    swin_result[
        "validation"
    ]["confusion_matrix"],
    "Swin Validation Confusion Matrix",
    SWIN_DIR / "confusion_matrix.png"
)


plot_confusion_matrix(
    swin_result[
        "validation"
    ]["confusion_matrix"],
    "Swin Normalized Validation Confusion Matrix",
    SWIN_DIR / "confusion_matrix_normalized.png",
    normalized=True
)


plot_probability_distribution(
    swin_result[
        "validation"
    ],
    "Swin Validation Prediction Distribution",
    SWIN_DIR / "prediction_distribution.png"
)


plot_misclassified_examples(
    swin_result[
        "validation"
    ],
    raw_val,
    bake_val_indices,
    "Swin Highest-Confidence Validation Errors",
    SWIN_DIR / "misclassified_examples.png"
)


gc.collect()
torch.cuda.empty_cache()


# ================================================================================================
# SECTION 22 — ARCHITECTURE COMPARISON
# ================================================================================================

comparison_rows = []


for result in [
    vit_result,
    swin_result
]:

    comparison_rows.append({
        "model":
            result[
                "architecture"
            ],

        "checkpoint":
            result[
                "model_name"
            ],

        "training_seconds":
            result[
                "training_seconds"
            ],

        **result[
            "metrics"
        ]
    })


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_df = (
    comparison_df
    .sort_values(
        "auc_fake",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


comparison_df.to_csv(
    COMPARE_DIR
    /
    "architecture_comparison.csv",
    index=False
)


print("\n" + "=" * 130)
print("SECTION 22 — ViT vs SWIN ARCHITECTURE COMPARISON")
print("=" * 130)


print(
    comparison_df[
        [
            "model",
            "accuracy",
            "balanced_accuracy",
            "precision_fake",
            "recall_fake",
            "specificity_real",
            "f1_fake",
            "auc_fake",
            "average_precision_fake",
            "kappa",
            "mcc",
            "training_seconds"
        ]
    ].to_string(
        index=False
    )
)


# ================================================================================================
# SECTION 23 — MAIN METRIC BAR GRAPH
# ================================================================================================

metric_names = [
    "accuracy",
    "balanced_accuracy",
    "precision_fake",
    "recall_fake",
    "specificity_real",
    "f1_fake",
    "auc_fake",
    "average_precision_fake",
    "mcc"
]


metric_labels = [
    "Accuracy",
    "Balanced\nAccuracy",
    "Fake\nPrecision",
    "Fake\nRecall",
    "Real\nSpecificity",
    "Fake F1",
    "ROC-AUC",
    "PR-AUC",
    "MCC"
]


fig, ax = plt.subplots(
    figsize=(15, 6)
)


x = np.arange(
    len(metric_names)
)

width = 0.36


for model_number, architecture in enumerate([
    "ViT",
    "Swin"
]):

    row = comparison_df[
        comparison_df["model"]
        ==
        architecture
    ].iloc[0]


    values = [
        float(
            row[metric]
        )
        for metric in metric_names
    ]


    positions = (
        x
        +
        (
            model_number
            -
            0.5
        )
        *
        width
    )


    bars = ax.bar(
        positions,
        values,
        width,
        label=architecture
    )


    for bar, value in zip(
        bars,
        values
    ):

        ax.text(
            bar.get_x()
            +
            bar.get_width() / 2,
            value + 0.01,
            f"{value:.3f}",
            ha="center",
            rotation=90,
            fontsize=8
        )


ax.set_xticks(x)
ax.set_xticklabels(metric_labels)

ax.set_ylim(
    0,
    1.12
)

ax.set_ylabel(
    "Score"
)

ax.set_title(
    "ViT vs Swin — Validation Performance"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    COMPARE_DIR / "metric_comparison.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 24 — TRAINING TIME COMPARISON
# ================================================================================================

training_times = [
    vit_result[
        "training_seconds"
    ] / 60,

    swin_result[
        "training_seconds"
    ] / 60
]


fig, ax = plt.subplots(
    figsize=(7, 5)
)

bars = ax.bar(
    [
        "ViT",
        "Swin"
    ],
    training_times
)

ax.set_ylabel(
    "Minutes"
)

ax.set_title(
    "Architecture Bake-Off Training Time"
)


for bar, value in zip(
    bars,
    training_times
):

    ax.text(
        bar.get_x()
        +
        bar.get_width() / 2,
        value,
        f"{value:.2f}",
        ha="center",
        va="bottom"
    )


fig.tight_layout()

fig.savefig(
    COMPARE_DIR / "training_time.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 25 — ROC COMPARISON
# ================================================================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)


for result in [
    vit_result,
    swin_result
]:

    evaluation = result[
        "validation"
    ]


    fpr, tpr, _ = roc_curve(
        evaluation[
            "binary_true"
        ],
        evaluation[
            "fake_probability"
        ]
    )


    ax.plot(
        fpr,
        tpr,
        linewidth=2,
        label=(
            f"{result['architecture']} "
            f"(AUC="
            f"{result['metrics']['auc_fake']:.4f})"
        )
    )


ax.plot(
    [
        0,
        1
    ],
    [
        0,
        1
    ],
    linestyle="--",
    label="Random"
)

ax.set_xlabel(
    "False Positive Rate"
)

ax.set_ylabel(
    "True Positive Rate"
)

ax.set_title(
    "ViT vs Swin — ROC Curves"
)

ax.legend(
    loc="lower right"
)

fig.tight_layout()

fig.savefig(
    COMPARE_DIR / "roc_comparison.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 26 — PRECISION-RECALL COMPARISON
# ================================================================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)


for result in [
    vit_result,
    swin_result
]:

    evaluation = result[
        "validation"
    ]


    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            evaluation[
                "binary_true"
            ],
            evaluation[
                "fake_probability"
            ]
        )
    )


    ax.plot(
        recall_curve,
        precision_curve,
        linewidth=2,
        label=(
            f"{result['architecture']} "
            f"(AP="
            f"{result['metrics']['average_precision_fake']:.4f})"
        )
    )


ax.set_xlabel(
    "Recall"
)

ax.set_ylabel(
    "Precision"
)

ax.set_title(
    "ViT vs Swin — Precision-Recall Curves"
)

ax.legend(
    loc="lower left"
)

fig.tight_layout()

fig.savefig(
    COMPARE_DIR
    /
    "precision_recall_comparison.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 27 — PERFORMANCE RADAR
# ================================================================================================

radar_metrics = [
    "accuracy",
    "precision_fake",
    "recall_fake",
    "specificity_real",
    "f1_fake",
    "auc_fake"
]


radar_labels = [
    "Accuracy",
    "Fake Precision",
    "Fake Recall",
    "Real Specificity",
    "Fake F1",
    "ROC-AUC"
]


angles = np.linspace(
    0,
    2 * np.pi,
    len(radar_metrics),
    endpoint=False
).tolist()

angles += angles[:1]


fig = plt.figure(
    figsize=(8, 8)
)

ax = fig.add_subplot(
    111,
    polar=True
)


for architecture in [
    "ViT",
    "Swin"
]:

    row = comparison_df[
        comparison_df["model"]
        ==
        architecture
    ].iloc[0]


    values = [
        float(
            row[metric]
        )
        for metric in radar_metrics
    ]

    values += values[:1]


    ax.plot(
        angles,
        values,
        linewidth=2,
        label=architecture
    )

    ax.fill(
        angles,
        values,
        alpha=0.10
    )


ax.set_xticks(
    angles[:-1]
)

ax.set_xticklabels(
    radar_labels
)

ax.set_ylim(
    0,
    1
)

ax.set_title(
    "ViT vs Swin Performance Radar"
)

ax.legend(
    loc="upper right",
    bbox_to_anchor=(
        1.25,
        1.10
    )
)

fig.tight_layout()

fig.savefig(
    COMPARE_DIR / "performance_radar.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 28 — SELECT WINNER
# ================================================================================================

WINNER_NAME = str(
    comparison_df.iloc[0][
        "model"
    ]
)

WINNER_MODEL_NAME = MODEL_NAMES[
    WINNER_NAME
]


print("\n" + "=" * 105)
print("SECTION 28 — WINNER SELECTION")
print("=" * 105)

print(
    "🏆 WINNING ARCHITECTURE:",
    WINNER_NAME
)

print(
    "Checkpoint:",
    WINNER_MODEL_NAME
)

print(
    "Selection metric: "
    "Validation fake-class ROC-AUC"
)

print(
    "✅ Test set has NOT been used."
)


# ================================================================================================
# SECTION 29 — FULL WINNER TRAINING
# ================================================================================================

print("\n" + "#" * 105)
print("SECTION 29 — FULL WINNER TRAINING")
print("#" * 105)

print(
    "Fresh pretrained model."
)

print(
    "Bake-off weights are NOT reused."
)


full_result = train_architecture(
    architecture=(
        WINNER_NAME
        +
        "_FULL"
    ),
    model_name=WINNER_MODEL_NAME,
    training_data=train_dataset,
    validation_data=val_dataset,
    epochs=FULL_EPOCHS,
    output_directory=FINAL_DIR,
    keep_model=True
)


# ================================================================================================
# SECTION 30 — FULL TRAINING LOSS CURVE
# ================================================================================================

plot_training_loss(
    full_result,
    FINAL_DIR
    /
    "full_training_step_loss.png"
)


history = full_result[
    "history"
]


fig, ax = plt.subplots(
    figsize=(8, 5)
)


ax.plot(
    history["epoch"],
    history["train_loss"],
    marker="o",
    label="Training Loss"
)


ax.plot(
    history["epoch"],
    history[
        "validation_loss"
    ],
    marker="o",
    label="Validation Loss"
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Loss"
)

ax.set_title(
    f"{WINNER_NAME} Full Training — Loss"
)

ax.set_xticks(
    history["epoch"]
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FINAL_DIR
    /
    "full_training_loss_by_epoch.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 31 — FULL VALIDATION PERFORMANCE
# ================================================================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)


ax.plot(
    history["epoch"],
    history[
        "validation_accuracy"
    ],
    marker="o",
    label="Accuracy"
)


ax.plot(
    history["epoch"],
    history[
        "validation_balanced_accuracy"
    ],
    marker="o",
    label="Balanced Accuracy"
)


ax.plot(
    history["epoch"],
    history[
        "validation_f1_fake"
    ],
    marker="o",
    label="Fake F1"
)


ax.plot(
    history["epoch"],
    history[
        "validation_auc_fake"
    ],
    marker="o",
    label="Fake ROC-AUC"
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Score"
)

ax.set_ylim(
    0,
    1.02
)

ax.set_xticks(
    history["epoch"]
)

ax.set_title(
    f"{WINNER_NAME} Full Validation Performance"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FINAL_DIR
    /
    "full_validation_metrics.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 32 — FULL VALIDATION CONFUSION MATRICES
# ================================================================================================

plot_confusion_matrix(
    full_result[
        "validation"
    ]["confusion_matrix"],
    f"{WINNER_NAME} Full Validation Confusion Matrix",
    FINAL_DIR
    /
    "full_validation_confusion_matrix.png"
)


plot_confusion_matrix(
    full_result[
        "validation"
    ]["confusion_matrix"],
    f"{WINNER_NAME} Normalized Full Validation Confusion Matrix",
    FINAL_DIR
    /
    "full_validation_confusion_matrix_normalized.png",
    normalized=True
)


# ================================================================================================
# SECTION 33 — FINAL UNTOUCHED TEST EVALUATION
# ================================================================================================

print("\n" + "=" * 105)
print("SECTION 33 — FINAL UNTOUCHED TEST EVALUATION")
print("=" * 105)

print(
    "This is the FIRST evaluation on the test set."
)

print(
    f"Test images: {len(test_dataset):,}"
)


test_loader = make_loader(
    test_dataset,
    EVAL_BATCH_SIZE,
    shuffle=False
)


final_model = full_result[
    "model"
].to(
    DEVICE
)


criterion = nn.CrossEntropyLoss()


test_result = evaluate_model(
    final_model,
    test_loader,
    criterion,
    description="FINAL TEST"
)


FINAL_METRICS = test_result[
    "metrics"
]


print("\n" + "=" * 105)
print("FINAL TEST RESULTS")
print("=" * 105)

print(
    "Architecture:",
    WINNER_NAME
)

print(
    "Checkpoint:",
    WINNER_MODEL_NAME
)

print()


for metric_name, metric_value in (
    FINAL_METRICS.items()
):

    print(
        f"{metric_name:28s}: "
        f"{metric_value:.6f}"
    )


# ================================================================================================
# SECTION 34 — SAVE TEST PREDICTIONS
# ================================================================================================

test_predictions_df = pd.DataFrame({
    "filepath":
        [
            path
            for path, _
            in raw_test.samples
        ],

    "true_id":
        test_result[
            "labels"
        ],

    "true_label":
        [
            CLASSES[
                int(value)
            ]
            for value
            in test_result[
                "labels"
            ]
        ],

    "predicted_id":
        test_result[
            "predictions"
        ],

    "predicted_label":
        [
            CLASSES[
                int(value)
            ]
            for value
            in test_result[
                "predictions"
            ]
        ],

    "probability_fake":
        test_result[
            "fake_probability"
        ],

    "probability_real":
        test_result[
            "probabilities"
        ][
            :,
            REAL_ID
        ]
})


test_predictions_df["correct"] = (
    test_predictions_df[
        "true_id"
    ]
    ==
    test_predictions_df[
        "predicted_id"
    ]
)


test_predictions_df.to_csv(
    FINAL_DIR
    /
    "final_test_predictions.csv",
    index=False
)


# ================================================================================================
# SECTION 35 — CLASSIFICATION REPORT
# ================================================================================================

report_dictionary = classification_report(
    test_result["labels"],
    test_result[
        "predictions"
    ],
    labels=[
        FAKE_ID,
        REAL_ID
    ],
    target_names=[
        "Fake",
        "Real"
    ],
    output_dict=True,
    zero_division=0
)


classification_df = pd.DataFrame(
    report_dictionary
).transpose()


classification_df.to_csv(
    FINAL_DIR
    /
    "final_classification_report.csv"
)


print(
    "\nFINAL CLASSIFICATION REPORT"
)

print(
    classification_df.to_string()
)


# ================================================================================================
# SECTION 36 — FINAL CONFUSION MATRICES
# ================================================================================================

plot_confusion_matrix(
    test_result[
        "confusion_matrix"
    ],
    f"{WINNER_NAME} Final Test Confusion Matrix",
    FINAL_DIR
    /
    "final_test_confusion_matrix.png"
)


plot_confusion_matrix(
    test_result[
        "confusion_matrix"
    ],
    f"{WINNER_NAME} Normalized Final Test Confusion Matrix",
    FINAL_DIR
    /
    "final_test_confusion_matrix_normalized.png",
    normalized=True
)


# ================================================================================================
# SECTION 37 — FINAL ROC CURVE
# ================================================================================================

final_fpr, final_tpr, _ = roc_curve(
    test_result[
        "binary_true"
    ],
    test_result[
        "fake_probability"
    ]
)


fig, ax = plt.subplots(
    figsize=(7, 6)
)


ax.plot(
    final_fpr,
    final_tpr,
    linewidth=2,
    label=(
        f"{WINNER_NAME} "
        f"(AUC="
        f"{FINAL_METRICS['auc_fake']:.5f})"
    )
)


ax.plot(
    [
        0,
        1
    ],
    [
        0,
        1
    ],
    linestyle="--",
    label="Random"
)


ax.set_xlabel(
    "False Positive Rate"
)

ax.set_ylabel(
    "True Positive Rate"
)

ax.set_title(
    "Final Unseen Test ROC Curve"
)

ax.legend(
    loc="lower right"
)

fig.tight_layout()

fig.savefig(
    FINAL_DIR
    /
    "final_test_roc_curve.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 38 — FINAL PRECISION-RECALL CURVE
# ================================================================================================

final_precision, final_recall, _ = (
    precision_recall_curve(
        test_result[
            "binary_true"
        ],
        test_result[
            "fake_probability"
        ]
    )
)


fig, ax = plt.subplots(
    figsize=(7, 6)
)


ax.plot(
    final_recall,
    final_precision,
    linewidth=2,
    label=(
        f"{WINNER_NAME} "
        f"(AP="
        f"{FINAL_METRICS['average_precision_fake']:.5f})"
    )
)


ax.set_xlabel(
    "Recall"
)

ax.set_ylabel(
    "Precision"
)

ax.set_title(
    "Final Unseen Test Precision-Recall Curve"
)

ax.legend(
    loc="lower left"
)

fig.tight_layout()

fig.savefig(
    FINAL_DIR
    /
    "final_test_precision_recall_curve.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 39 — FINAL PROBABILITY DISTRIBUTION
# ================================================================================================

plot_probability_distribution(
    test_result,
    "Final Test Predicted Fake-Probability Distribution",
    FINAL_DIR
    /
    "final_test_probability_distribution.png"
)


# ================================================================================================
# SECTION 40 — CALIBRATION GRAPH
# ================================================================================================

calibration_true, calibration_predicted = (
    calibration_curve(
        test_result[
            "binary_true"
        ],
        test_result[
            "fake_probability"
        ],
        n_bins=10,
        strategy="uniform"
    )
)


fig, ax = plt.subplots(
    figsize=(7, 6)
)


ax.plot(
    calibration_predicted,
    calibration_true,
    marker="o",
    label=WINNER_NAME
)


ax.plot(
    [
        0,
        1
    ],
    [
        0,
        1
    ],
    linestyle="--",
    label="Perfect Calibration"
)


ax.set_xlabel(
    "Mean Predicted Probability of Fake"
)

ax.set_ylabel(
    "Observed Fraction of Fake Images"
)

ax.set_title(
    "Final Test Calibration / Reliability Diagram"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FINAL_DIR
    /
    "final_test_calibration.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 41 — TEST MISCLASSIFICATIONS
# ================================================================================================

all_test_indices = list(
    range(
        len(raw_test)
    )
)


plot_misclassified_examples(
    test_result,
    raw_test,
    all_test_indices,
    f"{WINNER_NAME} Highest-Confidence Test Errors",
    FINAL_DIR
    /
    "final_test_misclassified_examples.png",
    maximum=12
)


# ================================================================================================
# SECTION 42 — HIGH-CONFIDENCE CORRECT EXAMPLES
# ================================================================================================

predictions = test_result[
    "predictions"
]

true_labels = test_result[
    "labels"
]

fake_probability = test_result[
    "fake_probability"
]


correct_positions = np.where(
    predictions
    ==
    true_labels
)[0]


confidence = np.where(
    predictions
    ==
    FAKE_ID,
    fake_probability,
    1
    -
    fake_probability
)


correct_positions = sorted(
    correct_positions,
    key=lambda position:
        confidence[position],
    reverse=True
)[:12]


fig, axes = plt.subplots(
    3,
    4,
    figsize=(14, 10)
)


axes = axes.flatten()


for axis in axes:
    axis.axis("off")


for plot_number, position in enumerate(
    correct_positions
):

    image_path, _ = raw_test.samples[
        int(position)
    ]


    with Image.open(
        image_path
    ) as image:

        image = image.convert(
            "RGB"
        )

        axes[
            plot_number
        ].imshow(
            image
        )


    class_name = CLASSES[
        int(
            true_labels[
                position
            ]
        )
    ]


    axes[
        plot_number
    ].set_title(
        f"Correct: {class_name}\n"
        f"P(fake): "
        f"{fake_probability[position]:.3f}"
    )


    axes[
        plot_number
    ].axis("off")


fig.suptitle(
    f"{WINNER_NAME} — "
    "High-Confidence Correct Test Predictions",
    fontsize=15
)

fig.tight_layout()

fig.savefig(
    FINAL_DIR
    /
    "final_test_correct_examples.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 43 — FINAL METRIC SUMMARY
# ================================================================================================

summary_metrics = {
    "Accuracy":
        FINAL_METRICS[
            "accuracy"
        ],

    "Balanced Accuracy":
        FINAL_METRICS[
            "balanced_accuracy"
        ],

    "Fake Precision":
        FINAL_METRICS[
            "precision_fake"
        ],

    "Fake Recall":
        FINAL_METRICS[
            "recall_fake"
        ],

    "Real Specificity":
        FINAL_METRICS[
            "specificity_real"
        ],

    "Fake F1":
        FINAL_METRICS[
            "f1_fake"
        ],

    "ROC-AUC":
        FINAL_METRICS[
            "auc_fake"
        ],

    "PR-AUC":
        FINAL_METRICS[
            "average_precision_fake"
        ],

    "MCC":
        FINAL_METRICS[
            "mcc"
        ]
}


fig, ax = plt.subplots(
    figsize=(12, 6)
)


bars = ax.bar(
    list(
        summary_metrics.keys()
    ),
    list(
        summary_metrics.values()
    )
)


ax.set_ylim(
    0,
    1.08
)

ax.set_ylabel(
    "Score"
)

ax.set_title(
    f"{WINNER_NAME} — Final Unseen Test Performance"
)

ax.tick_params(
    axis="x",
    rotation=45
)


for bar, value in zip(
    bars,
    summary_metrics.values()
):

    ax.text(
        bar.get_x()
        +
        bar.get_width() / 2,
        value + 0.01,
        f"{value:.4f}",
        ha="center"
    )


fig.tight_layout()

fig.savefig(
    FINAL_DIR
    /
    "final_test_metric_summary.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 44 — BOOTSTRAP 95% CONFIDENCE INTERVALS
# ================================================================================================

print("\n" + "=" * 105)
print("SECTION 44 — BOOTSTRAP 95% CONFIDENCE INTERVALS")
print("=" * 105)


rng = np.random.default_rng(
    SEED
)


test_labels = test_result[
    "labels"
]

test_predictions = test_result[
    "predictions"
]

binary_true = test_result[
    "binary_true"
]

binary_prediction = test_result[
    "binary_prediction"
]

test_fake_probabilities = test_result[
    "fake_probability"
]


number_of_test_images = len(
    test_labels
)


bootstrap_results = {
    "accuracy": [],
    "f1_fake": [],
    "auc_fake": [],
    "mcc": []
}


for _ in tqdm(
    range(
        BOOTSTRAP_ITERATIONS
    ),
    desc="Bootstrap confidence intervals"
):

    bootstrap_indices = rng.integers(
        0,
        number_of_test_images,
        size=number_of_test_images
    )


    bootstrap_binary_true = binary_true[
        bootstrap_indices
    ]


    if np.unique(
        bootstrap_binary_true
    ).size < 2:

        continue


    bootstrap_results[
        "accuracy"
    ].append(
        accuracy_score(
            test_labels[
                bootstrap_indices
            ],
            test_predictions[
                bootstrap_indices
            ]
        )
    )


    bootstrap_results[
        "f1_fake"
    ].append(
        f1_score(
            bootstrap_binary_true,
            binary_prediction[
                bootstrap_indices
            ],
            zero_division=0
        )
    )


    bootstrap_results[
        "auc_fake"
    ].append(
        roc_auc_score(
            bootstrap_binary_true,
            test_fake_probabilities[
                bootstrap_indices
            ]
        )
    )


    bootstrap_results[
        "mcc"
    ].append(
        matthews_corrcoef(
            bootstrap_binary_true,
            binary_prediction[
                bootstrap_indices
            ]
        )
    )


ci_rows = []


for metric_name, values in (
    bootstrap_results.items()
):

    values = np.asarray(
        values
    )


    lower = float(
        np.percentile(
            values,
            2.5
        )
    )


    upper = float(
        np.percentile(
            values,
            97.5
        )
    )


    ci_rows.append({
        "metric":
            metric_name,

        "point_estimate":
            FINAL_METRICS[
                metric_name
            ],

        "ci_95_lower":
            lower,

        "ci_95_upper":
            upper
    })


confidence_df = pd.DataFrame(
    ci_rows
)


confidence_df.to_csv(
    FINAL_DIR
    /
    "final_test_bootstrap_95ci.csv",
    index=False
)


print(
    confidence_df.to_string(
        index=False
    )
)


# ================================================================================================
# SECTION 45 — CONFIDENCE INTERVAL GRAPH
# ================================================================================================

points = confidence_df[
    "point_estimate"
].to_numpy()


lower_error = (
    points
    -
    confidence_df[
        "ci_95_lower"
    ].to_numpy()
)


upper_error = (
    confidence_df[
        "ci_95_upper"
    ].to_numpy()
    -
    points
)


fig, ax = plt.subplots(
    figsize=(8, 5)
)


positions = np.arange(
    len(
        confidence_df
    )
)


ax.errorbar(
    positions,
    points,
    yerr=np.vstack([
        lower_error,
        upper_error
    ]),
    fmt="o",
    capsize=5
)


ax.set_xticks(
    positions
)

ax.set_xticklabels(
    confidence_df[
        "metric"
    ]
)

ax.set_ylim(
    0,
    1.03
)

ax.set_ylabel(
    "Score"
)

ax.set_title(
    "Final Test Metrics with 95% Bootstrap Confidence Intervals"
)

fig.tight_layout()

fig.savefig(
    FINAL_DIR
    /
    "final_test_95ci.png",
    dpi=180
)

plt.show()
plt.close(fig)


# ================================================================================================
# SECTION 46 — SAVE FINAL MODEL
# ================================================================================================

FINAL_MODEL_PATH = (
    FINAL_DIR
    /
    "FINAL_WINNER_MODEL_STATE_DICT.pt"
)


torch.save(
    final_model.state_dict(),
    FINAL_MODEL_PATH
)


model_configuration = {
    "architecture":
        WINNER_NAME,

    "timm_model_name":
        WINNER_MODEL_NAME,

    "num_classes":
        2,

    "classes":
        CLASSES,

    "class_to_id":
        CLASS_TO_ID,

    "fake_id":
        FAKE_ID,

    "real_id":
        REAL_ID,

    "image_size":
        IMAGE_SIZE,

    "normalization_mean":
        IMAGENET_MEAN,

    "normalization_std":
        IMAGENET_STD
}


with open(
    FINAL_DIR
    /
    "FINAL_MODEL_CONFIG.json",
    "w"
) as file:

    json.dump(
        model_configuration,
        file,
        indent=2
    )


print(
    "\n✅ Final trained model saved."
)


# ================================================================================================
# SECTION 47 — MASTER RESULTS TABLE
# ================================================================================================

master_results = pd.DataFrame([
    {
        "experiment":
            "Architecture Bake-Off",

        "model":
            "ViT",

        "checkpoint":
            VIT_MODEL_NAME,

        **vit_result[
            "metrics"
        ]
    },

    {
        "experiment":
            "Architecture Bake-Off",

        "model":
            "Swin",

        "checkpoint":
            SWIN_MODEL_NAME,

        **swin_result[
            "metrics"
        ]
    },

    {
        "experiment":
            "Final Unseen Test",

        "model":
            WINNER_NAME,

        "checkpoint":
            WINNER_MODEL_NAME,

        **FINAL_METRICS
    }
])


master_results.to_csv(
    OUTPUT_ROOT
    /
    "MASTER_RESULTS.csv",
    index=False
)


# ================================================================================================
# SECTION 48 — FINAL JSON
# ================================================================================================

research_results = {
    "project":
        "ViT vs Swin Transformer for Real vs GAN-Generated Face Classification",

    "dataset":
        "140k Real and Fake Faces",

    "dataset_root":
        str(
            DATASET_ROOT
        ),

    "dataset_sizes": {
        "train":
            len(raw_train),

        "validation":
            len(raw_val),

        "test":
            len(raw_test)
    },

    "classes":
        CLASSES,

    "positive_class":
        "fake",

    "seed":
        SEED,

    "image_size":
        IMAGE_SIZE,

    "gpu":
        torch.cuda.get_device_name(0),

    "bakeoff": {
        "training_images":
            len(
                bake_train_dataset
            ),

        "validation_images":
            len(
                bake_val_dataset
            ),

        "epochs":
            BAKE_EPOCHS,

        "ViT": {
            "checkpoint":
                VIT_MODEL_NAME,

            "metrics":
                vit_result[
                    "metrics"
                ]
        },

        "Swin": {
            "checkpoint":
                SWIN_MODEL_NAME,

            "metrics":
                swin_result[
                    "metrics"
                ]
        }
    },

    "winner":
        WINNER_NAME,

    "winner_checkpoint":
        WINNER_MODEL_NAME,

    "selection_metric":
        "validation fake-class ROC-AUC",

    "full_training_epochs":
        FULL_EPOCHS,

    "full_validation_metrics":
        full_result[
            "metrics"
        ],

    "final_unseen_test_metrics":
        FINAL_METRICS
}


with open(
    FINAL_DIR
    /
    "FINAL_RESEARCH_RESULTS.json",
    "w"
) as file:

    json.dump(
        research_results,
        file,
        indent=2
    )


# ================================================================================================
# SECTION 49 — HUMAN-READABLE FINAL REPORT
# ================================================================================================

final_report = f"""
ViT vs Swin Transformer
Real vs GAN-Generated Face Classification
================================================================================

DATASET
--------------------------------------------------------------------------------
Dataset:
140k Real and Fake Faces

Dataset path:
{DATASET_ROOT}

Training images:
{len(raw_train):,}

Validation images:
{len(raw_val):,}

Test images:
{len(raw_test):,}

Classes:
{CLASSES}

Fake class ID:
{FAKE_ID}

Real class ID:
{REAL_ID}

Positive forensic class:
Fake


EXPERIMENTAL HARDWARE
--------------------------------------------------------------------------------
GPU:
{torch.cuda.get_device_name(0)}

PyTorch:
{torch.__version__}

timm:
{timm.__version__}


ARCHITECTURE BAKE-OFF
--------------------------------------------------------------------------------
Training subset:
{len(bake_train_dataset):,}

Validation subset:
{len(bake_val_dataset):,}

Epochs:
{BAKE_EPOCHS}


VISION TRANSFORMER
--------------------------------------------------------------------------------
Checkpoint:
{VIT_MODEL_NAME}

Accuracy:
{vit_result["metrics"]["accuracy"]:.6f}

Balanced Accuracy:
{vit_result["metrics"]["balanced_accuracy"]:.6f}

Fake Precision:
{vit_result["metrics"]["precision_fake"]:.6f}

Fake Recall:
{vit_result["metrics"]["recall_fake"]:.6f}

Real Specificity:
{vit_result["metrics"]["specificity_real"]:.6f}

Fake F1:
{vit_result["metrics"]["f1_fake"]:.6f}

Fake ROC-AUC:
{vit_result["metrics"]["auc_fake"]:.6f}

Fake PR-AUC:
{vit_result["metrics"]["average_precision_fake"]:.6f}

Cohen Kappa:
{vit_result["metrics"]["kappa"]:.6f}

MCC:
{vit_result["metrics"]["mcc"]:.6f}


SWIN TRANSFORMER
--------------------------------------------------------------------------------
Checkpoint:
{SWIN_MODEL_NAME}

Accuracy:
{swin_result["metrics"]["accuracy"]:.6f}

Balanced Accuracy:
{swin_result["metrics"]["balanced_accuracy"]:.6f}

Fake Precision:
{swin_result["metrics"]["precision_fake"]:.6f}

Fake Recall:
{swin_result["metrics"]["recall_fake"]:.6f}

Real Specificity:
{swin_result["metrics"]["specificity_real"]:.6f}

Fake F1:
{swin_result["metrics"]["f1_fake"]:.6f}

Fake ROC-AUC:
{swin_result["metrics"]["auc_fake"]:.6f}

Fake PR-AUC:
{swin_result["metrics"]["average_precision_fake"]:.6f}

Cohen Kappa:
{swin_result["metrics"]["kappa"]:.6f}

MCC:
{swin_result["metrics"]["mcc"]:.6f}


SELECTED ARCHITECTURE
--------------------------------------------------------------------------------
Winner:
{WINNER_NAME}

Checkpoint:
{WINNER_MODEL_NAME}

Selection criterion:
Validation fake-class ROC-AUC


FINAL UNSEEN TEST PERFORMANCE
--------------------------------------------------------------------------------
Accuracy:
{FINAL_METRICS["accuracy"]:.6f}

Balanced Accuracy:
{FINAL_METRICS["balanced_accuracy"]:.6f}

Fake Precision:
{FINAL_METRICS["precision_fake"]:.6f}

Fake Recall / Sensitivity:
{FINAL_METRICS["recall_fake"]:.6f}

Real Specificity:
{FINAL_METRICS["specificity_real"]:.6f}

Fake F1:
{FINAL_METRICS["f1_fake"]:.6f}

Fake ROC-AUC:
{FINAL_METRICS["auc_fake"]:.6f}

Fake PR-AUC:
{FINAL_METRICS["average_precision_fake"]:.6f}

Cohen Kappa:
{FINAL_METRICS["kappa"]:.6f}

Matthews Correlation Coefficient:
{FINAL_METRICS["mcc"]:.6f}

Brier Score:
{FINAL_METRICS["brier_fake"]:.6f}


SCIENTIFIC SCOPE
--------------------------------------------------------------------------------
This experiment evaluates classification of real face images versus
GAN-generated face images from the supplied 140k Real and Fake Faces dataset.

The results should therefore be described as performance on GAN-generated
face detection under this dataset and protocol. They should not be interpreted
as proof of universal detection of all deepfakes, face swaps, diffusion images,
video manipulations, or unseen generative models.

The selected pretrained ViT and Swin configurations also differ in parameter
count and pretrained-weight history. Therefore, the experiment compares the
selected pretrained architecture pipelines rather than a perfectly controlled
architecture-only ablation.
""".strip()


with open(
    FINAL_DIR
    /
    "FINAL_REPORT.txt",
    "w"
) as file:

    file.write(
        final_report
    )


print(
    "\n"
    +
    final_report
)


# ================================================================================================
# SECTION 50 — ZIP EVERYTHING
# ================================================================================================

ZIP_PATH = shutil.make_archive(
    "/kaggle/working/"
    "vit_vs_swin_research_complete",
    "zip",
    root_dir=str(
        OUTPUT_ROOT
    )
)


# ================================================================================================
# SECTION 51 — COMPLETION
# ================================================================================================

print("\n" + "=" * 110)
print("✅ COMPLETE RESEARCH PIPELINE FINISHED")
print("=" * 110)

print(
    "\n🏆 WINNER:"
)

print(
    WINNER_NAME
)

print(
    "\nFINAL MODEL:"
)

print(
    FINAL_MODEL_PATH
)

print(
    "\nMASTER RESULTS:"
)

print(
    OUTPUT_ROOT
    /
    "MASTER_RESULTS.csv"
)

print(
    "\nALL OUTPUTS:"
)

print(
    OUTPUT_ROOT
)

print(
    "\nZIP FILE:"
)

print(
    ZIP_PATH
)

print(
    "\nGENERATED SECTIONS:"
)

print(
    " • Dataset distribution"
)

print(
    " • Dataset image samples"
)

print(
    " • ViT parameter/model information"
)

print(
    " • ViT training-loss graph"
)

print(
    " • ViT confusion matrices"
)

print(
    " • ViT prediction-confidence graph"
)

print(
    " • ViT error examples"
)

print(
    " • Swin parameter/model information"
)

print(
    " • Swin training-loss graph"
)

print(
    " • Swin confusion matrices"
)

print(
    " • Swin prediction-confidence graph"
)

print(
    " • Swin error examples"
)

print(
    " • ViT-vs-Swin metric comparison"
)

print(
    " • Training-time comparison"
)

print(
    " • Parameter-count comparison"
)

print(
    " • ROC comparison"
)

print(
    " • Precision-recall comparison"
)

print(
    " • Radar comparison"
)

print(
    " • Winner full-training loss graph"
)

print(
    " • Winner validation graph"
)

print(
    " • Final test classification report"
)

print(
    " • Final test confusion matrices"
)

print(
    " • Final test ROC curve"
)

print(
    " • Final test precision-recall curve"
)

print(
    " • Final test probability distribution"
)

print(
    " • Final calibration graph"
)

print(
    " • Test misclassification visualization"
)

print(
    " • High-confidence correct predictions"
)

print(
    " • Final metric-summary graph"
)

print(
    " • Bootstrap 95% confidence intervals"
)

print(
    " • Final model weights"
)

print(
    " • Model configuration"
)

print(
    " • CSV results"
)

print(
    " • JSON results"
)

print(
    " • Human-readable research report"
)

print(
    " • Complete ZIP archive"
)

print("\n" + "=" * 110)
print("END")
print("=" * 110)